## Configuration-Driven Satellite Loading

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()

while not (project_root / "src").is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not find project root")

    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [2]:
import geopandas as gpd
import geemap
import ee

ee.Authenticate()
ee.Initialize()

print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [3]:
from src.areas import load_zones, validate_zones

from src.config import load_config, validate_config

from src.satellite import (
    load_landsat_from_config,

    get_collection_size,
    get_collection_date_range,
    get_first_image,

    get_available_sensors,
    get_sensor_type,
    get_band_mapping,
    get_server_side_band_mapping,

    validate_collection,
)

from src.preprocessing import (
    mask_clouds,
    mask_snow,
    apply_reflectance_scaling,
    preprocess_collection
)

from src.indices import add_ndvi, add_time_metadata, add_ndvi_by_sensor, create_annual_ndvi_composite, create_annual_ndvi_collection

In [4]:
# Project root
PROJECT_ROOT = Path.cwd().parent

# Configuration file
config_path = PROJECT_ROOT / "config" / "settings.yaml"

# Load and validate configuration
config = load_config(config_path)

validate_config(config)

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [5]:
zones = load_zones()

validate_zones(zones)

print(f"Number of zones: {len(zones)}")
print(zones[["zone_id", "zone_type", "name"]])

study_area = zones.union_all()

study_area_geojson = study_area.__geo_interface__

study_geometry = ee.Geometry(study_area_geojson)

Number of zones: 4
             zone_id     zone_type                                 name
0  aoi_leh_immediate           aoi  Leh town and immediate surroundings
1           urban_01         urban                 Leh urban settlement
2     agriculture_01  agricultural          Irrigated agricultural land
3         natural_01       natural                     Natural mountain


In [6]:
landsat_study = load_landsat_from_config(
    study_geometry=study_geometry,
    config=config,
)

print(
    "Collection valid:",
    validate_collection(landsat_study)
)

print(
    "Number of images:",
    get_collection_size(landsat_study)
)

Collection valid: True
Number of images: 647


## Satellite Metadata Inspection

In [7]:
first_image = get_first_image(
    landsat_study
)

print("Sensor type:", get_sensor_type(first_image))
print("Band mapping:", get_band_mapping(first_image))

print("Date range:", get_collection_date_range(landsat_study))
print("Available sensors:", get_available_sensors(landsat_study))

Sensor type: landsat_457
Band mapping: {'blue': 'SR_B1', 'green': 'SR_B2', 'red': 'SR_B3', 'nir': 'SR_B4', 'swir1': 'SR_B5', 'swir2': 'SR_B7'}
Date range: {'start_date': '1989-08-06', 'end_date': '2025-12-23'}
Available sensors: ['LANDSAT_5', 'LANDSAT_7', 'LANDSAT_8', 'LANDSAT_9']


## Preprocessing pipeline

In [8]:
# Apply the complete preprocessing pipeline.
landsat_preprocessed = preprocess_collection(
    landsat_study,
    "landsat"
)

print("Preprocessing pipeline executed.")
print(landsat_preprocessed)

Preprocessing pipeline executed.
ee.ImageCollection({
  "functionInvocationValue": {
    "functionName": "Collection.map",
    "arguments": {
      "baseAlgorithm": {
        "functionDefinitionValue": {
          "argumentNames": [
            "_MAPPING_VAR_0_0"
          ],
          "body": {
            "functionInvocationValue": {
              "functionName": "Image.addBands",
              "arguments": {
                "dstImg": {
                  "argumentReference": "_MAPPING_VAR_0_0"
                },
                "overwrite": {
                  "constantValue": true
                },
                "srcImg": {
                  "functionInvocationValue": {
                    "functionName": "Image.add",
                    "arguments": {
                      "image1": {
                        "functionInvocationValue": {
                          "functionName": "Image.multiply",
                          "arguments": {
                            "image1": {
   

## NDVI calculation

In [9]:
# Get one image from the preprocessed collection.
image = get_first_image(landsat_preprocessed)

# Get the appropriate band mapping.
band_mapping = get_band_mapping(image)

# Calculate NDVI.
image_with_ndvi = add_ndvi(
    image=image,
    red_band=band_mapping["red"],
    nir_band=band_mapping["nir"],
)

# Inspect the bands.
print(image_with_ndvi.bandNames().getInfo())

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'SR_ATMOS_OPACITY', 'SR_CLOUD_QA', 'ST_B6', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'NDVI']


In [10]:
# Add date metadata to the image.
image_with_metadata = add_time_metadata(
    image_with_ndvi
)

# Inspect the metadata.
print(
    image_with_metadata
    .toDictionary([
        "year",
        "month",
        "day_of_year",
        "acquisition_date"
    ])
    .getInfo()
)

{'acquisition_date': '1989-08-06', 'day_of_year': 217, 'month': 8, 'year': 1989}


In [11]:
# Filter the existing preprocessed collection to 1989.
landsat_1989 = landsat_preprocessed.filterDate(
    "1989-05-01",
    "1989-11-01"
)

# Add NDVI to every image.
ndvi_1989 = landsat_1989.map(
    lambda image: add_ndvi(
        image,
        red_band="SR_B3",
        nir_band="SR_B4"
    )
)

# Add time metadata to every image.
ndvi_1989 = ndvi_1989.map(
    add_time_metadata
)

print(
    "Images in 1989:",
    ndvi_1989.size().getInfo()
)

Images in 1989: 1


In [12]:

# Create the annual median NDVI composite.
ndvi_1989_composite = (
    ndvi_1989
    .select("NDVI")
    .median()
    .rename("NDVI")
)

print(
    ndvi_1989_composite.bandNames().getInfo()
)

['NDVI']


In [13]:

# Calculate basic statistics for the 1989 composite.
ndvi_stats = ndvi_1989_composite.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=study_geometry,
    scale=30,
    maxPixels=1e9
)

print(ndvi_stats.getInfo())

{'NDVI_max': 0.7991679310798645, 'NDVI_min': -0.2466839849948883}


In [14]:

# Inspect the 1989 image collection.

print("Images in 1989:", landsat_1989.size().getInfo())

# Print acquisition dates.
dates_1989 = landsat_1989.aggregate_array(
    "system:time_start"
).getInfo()

print("Acquisition dates:")

for timestamp in dates_1989:
    print(
        ee.Date(timestamp)
        .format("YYYY-MM-dd")
        .getInfo()
    )

Images in 1989: 1
Acquisition dates:
1989-08-06


In [15]:
# Apply sensor-aware NDVI to the 1989 collection.
ndvi_1989_sensor_aware = landsat_1989.map(
    add_ndvi_by_sensor
)

print(
    "Images:",
    ndvi_1989_sensor_aware.size().getInfo()
)

print(
    "Bands:",
    ndvi_1989_sensor_aware
    .first()
    .bandNames()
    .getInfo()
)

Images: 1
Bands: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'SR_ATMOS_OPACITY', 'SR_CLOUD_QA', 'ST_B6', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'NDVI']


## Testing NDVI calculation on mix sensor data in 2020

In [16]:
# Get images from 2020.
landsat_2020 = landsat_preprocessed.filterDate(
    "2020-05-01",
    "2020-11-01"
)

print(
    "Images in 2020:",
    landsat_2020.size().getInfo()
)

Images in 2020: 13


In [17]:
# Get one image from the 2020 collection.
image_2020 = get_first_image(landsat_2020)

# Inspect the sensor.
print(
    "Sensor:",
    get_sensor_type(image_2020)
)

# Inspect the server-side band mapping.
mapping_2020 = get_server_side_band_mapping(
    image_2020
)

print(
    "Band mapping:",
    mapping_2020.getInfo()
)

Sensor: landsat_457
Band mapping: {'blue': 'SR_B1', 'green': 'SR_B2', 'nir': 'SR_B4', 'red': 'SR_B3', 'swir1': 'SR_B5', 'swir2': 'SR_B7'}


In [18]:
# Inspect the spacecraft IDs in the 2020 collection.

spacecraft_ids_2020 = landsat_2020.aggregate_array(
    "SPACECRAFT_ID"
).getInfo()

print(
    "Spacecraft IDs:",
    spacecraft_ids_2020
)

print(
    "Unique sensors:",
    set(spacecraft_ids_2020)
)

Spacecraft IDs: ['LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_7', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8', 'LANDSAT_8']
Unique sensors: {'LANDSAT_8', 'LANDSAT_7'}


In [19]:
# Apply it to the entire 2020 collection.
ndvi_2020_sensor_aware = landsat_2020.map(
    add_ndvi_by_sensor
)

# Check the number of images.
print(
    "Images:",
    ndvi_2020_sensor_aware.size().getInfo()
)

Images: 13


In [20]:
# Select only Landsat 8 images from 2020.

landsat_8_2020 = landsat_2020.filter(
    ee.Filter.eq(
        "SPACECRAFT_ID",
        "LANDSAT_8"
    )
)

print(
    "Landsat 8 images:",
    landsat_8_2020.size().getInfo()
)

Landsat 8 images: 7


In [21]:
# Get one Landsat 8 image.
image_landsat_8 = get_first_image(
    landsat_8_2020
)

# Inspect the server-side band mapping.
mapping_landsat_8 = get_server_side_band_mapping(
    image_landsat_8
)

print(
    mapping_landsat_8.getInfo()
)

{'blue': 'SR_B2', 'green': 'SR_B3', 'nir': 'SR_B5', 'red': 'SR_B4', 'swir1': 'SR_B6', 'swir2': 'SR_B7'}


In [22]:
# Apply sensor-aware NDVI to Landsat 8 images.
ndvi_landsat_8_2020 = landsat_8_2020.map(
    add_ndvi_by_sensor
)

# Inspect the first image's bands.
print(
    ndvi_landsat_8_2020
    .first()
    .bandNames()
    .getInfo()
)

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'NDVI']


## Annual Composite creation

In [23]:
# Create the 1989 annual NDVI composite.

ndvi_1989_annual = create_annual_ndvi_composite(
    collection=landsat_preprocessed,
    year=1989,
    start_month=5,
    end_month=10,
)

print("Annual composite created.")

# Inspect the annual composite.

print(
    "Bands:",
    ndvi_1989_annual.bandNames().getInfo()
)

print(
    "Metadata:",
    ndvi_1989_annual.toDictionary([
        "year",
        "start_month",
        "end_month",
        "image_count",
    ]).getInfo()
)

# Calculate statistics for the 1989 annual composite.

annual_stats_1989 = ndvi_1989_annual.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=study_geometry,
    scale=30,
    maxPixels=1e9,
)

print(
    annual_stats_1989.getInfo()
)

Annual composite created.
Bands: ['NDVI']
Metadata: {'end_month': 10, 'image_count': 1, 'start_month': 5, 'year': 1989}
{'NDVI_max': 0.7991679310798645, 'NDVI_min': -0.2466839849948883}


In [24]:
# Create the 2020 annual NDVI composite.

ndvi_2020_annual = create_annual_ndvi_composite(
    collection=landsat_preprocessed,
    year=2020,
    start_month=5,
    end_month=10,
)

print("2020 annual composite created.")

# Verify the 2020 annual composite.

print(
    "Bands:",
    ndvi_2020_annual.bandNames().getInfo()
)

print(
    "Metadata:",
    ndvi_2020_annual.toDictionary([
        "year",
        "start_month",
        "end_month",
        "image_count",
    ]).getInfo()
)

# Calculate NDVI statistics.
annual_stats_2020 = ndvi_2020_annual.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=study_geometry,
    scale=30,
    maxPixels=1e9,
)

print(
    "NDVI statistics:",
    annual_stats_2020.getInfo()
)

2020 annual composite created.
Bands: ['NDVI']
Metadata: {'end_month': 10, 'image_count': 13, 'start_month': 5, 'year': 2020}
NDVI statistics: {'NDVI_max': 0.7412186861038208, 'NDVI_min': -0.20922008156776428}


## Full NDVI composite collection

In [25]:
# Test multi-year NDVI processing.

annual_ndvi_test = create_annual_ndvi_collection(
    collection=landsat_preprocessed,
    years=[1989, 2020],
    start_month=5,
    end_month=10,
)

print(
    "Number of annual composites:",
    annual_ndvi_test.size().getInfo()
)

print(
    "Years:",
    annual_ndvi_test
    .aggregate_array("year")
    .getInfo()
)

Number of annual composites: 2
Years: [1989, 2020]


In [26]:
# Get all acquisition timestamps.
timestamps = landsat_preprocessed.aggregate_array(
    "system:time_start"
)

# Convert timestamps to years.
available_years = (
    ee.List(timestamps)
    .map(lambda timestamp: ee.Date(timestamp).get("year"))
    .distinct()
    .sort()
    .getInfo()
)

print("Available years:")
print(available_years)

print(
    "Number of available years:",
    len(available_years)
)

Available years:
[1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Number of available years: 37


In [27]:
# Create annual NDVI composites for all available years.

annual_ndvi_collection = create_annual_ndvi_collection(
    collection=landsat_preprocessed,
    years=available_years,
    start_month=5,
    end_month=10,
)

print(
    "Annual composites:",
    annual_ndvi_collection.size().getInfo()
)

Annual composites: 37


In [28]:
# Verify the annual NDVI collection.

print(
    "Bands of first composite:",
    annual_ndvi_collection
    .first()
    .bandNames()
    .getInfo()
)

print(
    "Composite years:",
    annual_ndvi_collection
    .aggregate_array("year")
    .getInfo()
)

print(
    "Image counts per year:",
    annual_ndvi_collection
    .aggregate_array("image_count")
    .getInfo()
)

Bands of first composite: ['NDVI']
Composite years: [1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Image counts per year: [1, 3, 2, 3, 2, 3, 0, 4, 1, 3, 2, 6, 3, 3, 3, 2, 4, 6, 5, 11, 15, 8, 13, 4, 12, 11, 10, 14, 16, 10, 15, 13, 12, 30, 27, 18, 12]


In [29]:
# Identify years with no images in the growing season.

composite_years = annual_ndvi_collection.aggregate_array(
    "year"
).getInfo()

image_counts = annual_ndvi_collection.aggregate_array(
    "image_count"
).getInfo()

zero_image_years = [
    year
    for year, count in zip(composite_years, image_counts)
    if count == 0
]

print("Years with zero images:")
print(zero_image_years)

print(
    "Number of zero-image years:",
    len(zero_image_years)
)

Years with zero images:
[1995]
Number of zero-image years: 1


In [31]:
# Keep only annual composites that contain at least one image.

annual_ndvi_valid = annual_ndvi_collection.filter(
    ee.Filter.gt("image_count", 0)
)

print(
    "Valid annual composites:",
    annual_ndvi_valid.size().getInfo()
)

print(
    "Excluded annual composites:",
    annual_ndvi_collection.size().getInfo()
    - annual_ndvi_valid.size().getInfo()
)

Valid annual composites: 36
Excluded annual composites: 1


In [34]:
# Check the number of valid annual composites.
print(
    "Valid annual composites:",
    annual_ndvi_valid.size().getInfo()
)

# Check the available years.
valid_years = (
    annual_ndvi_valid
    .aggregate_array("year")
    .getInfo()
)

print("Valid years:")
print(valid_years)

# Check the first and last year.
print("First year:", min(valid_years))
print("Last year:", max(valid_years))

# Check the bands of the first annual composite.
print(
    "First composite bands:",
    annual_ndvi_valid
    .first()
    .bandNames()
    .getInfo()
)

# Check the metadata of the first composite.
print(
    "First composite metadata:",
    annual_ndvi_valid
    .first()
    .toDictionary([
        "year",
        "start_month",
        "end_month",
        "image_count"
    ])
    .getInfo()
)

Valid annual composites: 36
Valid years:
[1989, 1990, 1991, 1992, 1993, 1994, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
First year: 1989
Last year: 2025
First composite bands: ['NDVI']
First composite metadata: {'end_month': 10, 'image_count': 1, 'start_month': 5, 'year': 1989}
